### Zero-shot children validation example

In [ ]:
import os
import pandas as pd
import tqdm
import random
import time

from utils.rag import (
    expand_setups,
)

from utils.utils import format_subsets_ontologies_paths
from utils.onto_object import OntologyEntryAttr, OntologyAccess

from modules.llm_validator import LLMValidator
from modules.evaluator import OntologyAlignmentEvaluator


# -------------------------
# DATASETS
# -------------------------

ALL_DATASET_NAMES = {
    # "anatomy": ["human-mouse"],
    # "bioml-2024": ["snomed-fma.body", "snomed-ncit.neoplas", "snomed-ncit.pharm", "ncit-doid", "omim-ordo"],
    "bioml-2024": ["omim-ordo"],
    # "largebio": ["fma-snomed", "snomed-nci", "fma-nci"],
    # "largebio_small": ["fma-nci", "snomed-nci", "fma-nci"],
}


# -------------------------
# EXPERIMENT SETUPS
# -------------------------

BASE_SETUP = {
    "subset": "omim-ordo",
    "model": "meta-llama/llama-3.1-70b-instruct",
    "max_workers": 1,
    "suffix": "_reduced",
    "prompt_functions": ["prompt_direct_entity_children", "prompt_direct_entity_children_no_parents", "direct_entity"],
}

SETUPS = [
    {"experiment_type": "zero_shot", "k": 0},
]

SETUPS = expand_setups(SETUPS, BASE_SETUP)

print("Expanded setups:")
for s in SETUPS:
    print(s)


# -------------------------
# RUN EXPERIMENTS
# -------------------------

for dataset_name, subfolders in ALL_DATASET_NAMES.items():

    for subfolder in subfolders:

        print(f"\nRunning dataset {dataset_name}/{subfolder}")

        o1_path, o2_path = format_subsets_ontologies_paths(dataset_name, subfolder)

        gt_path = f"data/{dataset_name}/{subfolder}/refs_equiv/full.tsv"

        onto_src = OntologyAccess(o1_path, annotate_on_init=True)
        onto_tgt = OntologyAccess(o2_path, annotate_on_init=True)

        validator = LLMValidator()

        with open(
            f"data/{dataset_name}/{subfolder}/{dataset_name}-{subfolder}-oasystem_mappings_to_ask_oracle_user_llm.txt",
            "r"
        ) as f:
            lines = f.readlines()
        

        for setup in SETUPS:

            experiment_type = setup["experiment_type"]
            k = setup["k"]
            model = setup["model"]

            prompt_type = setup["prompt_functions"]
            print(prompt_type)
            prompt_key = f"prompt_{prompt_type}"

            print(
                f"\nRunning setup | dataset={dataset_name} "
                f"| type={experiment_type} | k={k} | model={model}"
            )


            few_shot_examples = None
            vectorstore = None

            results = []

            # -------------------------
            # VALIDATION LOOP
            # -------------------------

            for line in tqdm.tqdm(
                lines,
                desc=f"{experiment_type}_k{k}"
            ):

                src_uri, tgt_uri = line.strip().split("|")[0], line.strip().split("|")[1]

                src_entity = OntologyEntryAttr(class_uri=src_uri, onto=onto_src)
                tgt_entity = OntologyEntryAttr(class_uri=tgt_uri, onto=onto_tgt)

                # -------------------------
                # ZERO SHOT
                # -------------------------

                if experiment_type == "zero_shot":
                    res = validator.validate(
                        src_entity,
                        tgt_entity,
                        prompt_type=prompt_type,
                        model=model,
                    )
                else:
                    raise ValueError(f"Unknown experiment type {experiment_type}")

                results.append({
                    "Source": src_uri,
                    "Target": tgt_uri,
                    "LLMDecision": res["is_match"],
                    "LLMConfidence": res["confidence"],
                    "LLMTotalTokens": res["tokens_used"],
                    "ExperimentType": experiment_type,
                    "K": k,
                    "Model": model
                })

            # -------------------------
            # SAVE RESULTS
            # -------------------------

            df = pd.DataFrame(results)

            output_dir = f"output/results_few_shot/{dataset_name}/{experiment_type}_k{k}"
            os.makedirs(output_dir, exist_ok=True)

            df.to_csv(f"{output_dir}/predictions.csv", index=False)

            # -------------------------
            # EVALUATION
            # -------------------------

            evaluator = OntologyAlignmentEvaluator(gt_path)

            report = evaluator.evaluate(
                df=df,
                dataset_name=dataset_name,
                experiment_type=f"{experiment_type}_k{k}",
                prompts_used=prompt_type,
                results_dir="output/results"
            )

            print("\n=== Evaluation Report ===")
            print(report)


### Run Few-Shot Validation

Note: tested on direct entity

In [ ]:
import os
import pandas as pd
import tqdm
import random
import time

from utils.rag import (
    expand_setups,
    load_few_shot_examples,
    build_vectorstore,
)

from utils.utils import format_subsets_ontologies_paths
from utils.onto_object import OntologyEntryAttr, OntologyAccess

from modules.llm_validator import LLMValidator
from modules.evaluator import OntologyAlignmentEvaluator


# -------------------------
# DATASETS
# -------------------------

ALL_DATASET_NAMES = {
    "anatomy": ["human-mouse"],
}


# -------------------------
# EXPERIMENT SETUPS
# -------------------------

BASE_SETUP = {
    "subset": "human-mouse",
    "model": "qwen/qwen3-vl-8b-instruct",
    "max_workers": 1,
    "suffix": "_reduced",
    "experiment_name": "v0",
    "prompt_functions": ["direct_entity"],
}

SETUPS = [
    # {"experiment_type": "zero_shot", "k": 0},
    {"experiment_type": "few_shot", "k": [1, 3, 5, 10]},
    # {"experiment_type": "few_shot_rag", "k": [5, 10]},
]

SETUPS = expand_setups(SETUPS, BASE_SETUP)

print("Expanded setups:")
for s in SETUPS:
    print(s)


# -------------------------
# RUN EXPERIMENTS
# -------------------------

for dataset_name, subfolders in ALL_DATASET_NAMES.items():

    for subfolder in subfolders:

        print(f"\nRunning dataset {dataset_name}/{subfolder}")

        o1_path, o2_path = format_subsets_ontologies_paths(dataset_name, subfolder)

        gt_path = f"data/{dataset_name}/{subfolder}/refs_equiv/full.tsv"

        onto_src = OntologyAccess(o1_path, annotate_on_init=True)
        onto_tgt = OntologyAccess(o2_path, annotate_on_init=True)

        validator = LLMValidator()

        with open(
            f"data/{dataset_name}/{subfolder}/{dataset_name}-{subfolder}-oasystem_mappings_to_ask_oracle_user_llm.txt",
            "r"
        ) as f:
            lines = f.readlines()
        

        for setup in SETUPS:

            experiment_type = setup["experiment_type"]
            k = setup["k"]
            model = setup["model"]

            prompt_type = setup["prompt_functions"]
            print(prompt_type)
            prompt_key = f"prompt_{prompt_type}"

            print(
                f"\nRunning setup | dataset={dataset_name} "
                f"| type={experiment_type} | k={k} | model={model}"
            )

            # -------------------------
            # FEW SHOT DATA
            # -------------------------

            few_shot_examples = None
            vectorstore = None

            if experiment_type in ["few_shot", "few_shot_rag"]:

                few_shot_path = f"data/preparation/{dataset_name}/{subfolder}/{setup['experiment_name']}.json"

                few_shot_examples = load_few_shot_examples(
                    few_shot_path,
                    prompt_key
                )

                # print(few_shot_examples)

                if len(few_shot_examples) == 0:
                    raise ValueError(f"No few-shot examples found for {prompt_key}")

                if experiment_type == "few_shot_rag":
                    vectorstore = build_vectorstore(few_shot_examples)
                    # print(vectorstore.docstore._dict)

            results = []

            # -------------------------
            # VALIDATION LOOP
            # -------------------------

            for line in tqdm.tqdm(
                lines,
                desc=f"{experiment_type}_k{k}"
            ):

                src_uri, tgt_uri = line.strip().split("|")[0], line.strip().split("|")[1]

                src_entity = OntologyEntryAttr(class_uri=src_uri, onto=onto_src)
                tgt_entity = OntologyEntryAttr(class_uri=tgt_uri, onto=onto_tgt)

                # -------------------------
                # ZERO SHOT
                # -------------------------

                if experiment_type == "zero_shot":
                    res = validator.validate(
                        src_entity,
                        tgt_entity,
                        prompt_type=prompt_type,
                        model=model,
                    )

                # -------------------------
                # FEW SHOT
                # -------------------------

                elif experiment_type == "few_shot":

                    # examples_subset = random.sample(
                    #     few_shot_examples,
                    #     min(k, len(few_shot_examples))
                    # )

                    # res = validator.validate_few_shot(
                    #     src_entity,
                    #     tgt_entity,
                    #     few_shot_examples=examples_subset,
                    #     k=len(examples_subset),
                    #     prompt_type=prompt_type,
                    #     model=model,
                    # )

                    res = validator.validate_few_shot(
                        src_entity,
                        tgt_entity,
                        few_shot_examples=few_shot_examples,
                        k=k,
                        prompt_type=prompt_type,
                        model=model,
                    )
  
                # -------------------------
                # FEW SHOT RAG
                # -------------------------

                elif experiment_type == "few_shot_rag":

                    res = validator.validate_few_shot_rag(
                        src_entity,
                        tgt_entity,
                        vectorstore=vectorstore,
                        few_shot_examples=few_shot_examples,
                        k=k,
                        prompt_type=prompt_type,
                        model=model,
                    )
                    print(res)
                else:
                    raise ValueError(f"Unknown experiment type {experiment_type}")

                results.append({
                    "Source": src_uri,
                    "Target": tgt_uri,
                    "LLMDecision": res["is_match"],
                    "LLMConfidence": res["confidence"],
                    "LLMTotalTokens": res["tokens_used"],
                    "ExperimentType": experiment_type,
                    "K": k,
                    "Model": model
                })

            # -------------------------
            # SAVE RESULTS
            # -------------------------

            df = pd.DataFrame(results)

            output_dir = f"output/results_few_shot/{dataset_name}/{experiment_type}_k{k}"
            os.makedirs(output_dir, exist_ok=True)

            df.to_csv(f"{output_dir}/predictions.csv", index=False)

            # -------------------------
            # EVALUATION
            # -------------------------

            evaluator = OntologyAlignmentEvaluator(gt_path)

            report = evaluator.evaluate(
                df=df,
                dataset_name=dataset_name,
                experiment_type=f"{experiment_type}_k{k}",
                prompts_used=prompt_type,
                results_dir="output/results_few_shot"
            )

            print("\n=== Evaluation Report ===")
            print(report)